In [37]:
# --- Notebook walkthrough: environment ---
# This notebook runs in Google Colab or local Jupyter. Printing `os.getcwd()` helps students
# find uploaded files (Colab often uses `/content`; local paths follow the project folder).

# Google Colab-ready setup
# This notebook is configured to run in Colab or another Jupyter environment.

import os
import numpy as np  # Numeric arrays / quick stats if you extend exercises with tabular data.
import pandas as pd  # Tables and CSV I/O—optional for the core LLM demos below.

print("If you upload files in Colab, they are usually available under /content.")


If you upload files in Colab, they are usually available under /content.


### Setup
#### Load the API key, install dependencies, and define basic functions for Google Colab.

In Colab, add `HKUST_Azure_API` in the **Secrets** panel, or enter it manually when prompted.

In [2]:
# --- Notebook walkthrough: install OpenAI SDK ---
# `%pip` installs into the *current* Jupyter kernel so `import openai` works in the next cells.

%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


Follow the instructions in https://itso.hkust.edu.hk/services/it-infrastructure/azure-openai-api-service to setup your account and api key.

In [ ]:
# --- Notebook walkthrough: Azure OpenAI client ---
# This cell connects the notebook to HKUST's Azure-hosted OpenAI service.
# We never hard-code API keys in source code; we load them safely from Colab Secrets,
# environment variables, or an interactive prompt—same idea you should use in real projects.

import os
from getpass import getpass
from openai import AzureOpenAI

# Colab provides `userdata` for named secrets. On a local Jupyter machine that module
# does not exist, so we fall back to `userdata = None` and use the other key sources.
try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_api_key(secret_name="HKUST_Azure_API"):
    """Resolve the API key in a portable order: Colab Secret → env var → typed input."""
    # 1) Google Colab: read from the Secrets store (recommended in class if you use Colab).
    if userdata is not None:
        try:
            value = userdata.get(secret_name)
            if value:
                return value
        except Exception:
            pass

    # 2) Local / server: standard practice is `export HKUST_Azure_API=...` (or .env loaded into the env).
    value = os.getenv(secret_name)
    if value:
        return value

    # 3) Last resort: ask once in the terminal/UI without echoing the key (getpass).
    return getpass(f"Enter {secret_name}: ")


# Load credentials and build one shared client object for the rest of the notebook.
secret_value_0 = get_api_key("HKUST_Azure_API")

# AzureOpenAI needs the resource endpoint URL, your key, and an API version string
# that matches what the gateway supports (HKUST documents which version to use).
client = AzureOpenAI(
    azure_endpoint="https://hkust.azure-api.net",  # HKUST Azure endpoint
    api_key=secret_value_0,
    api_version="2025-02-01-preview",
)

In [4]:
# --- Notebook walkthrough: chat helper functions ---
# `send_message` centralizes calls to `client.chat.completions.create`.
# `_normalize_temperature` reflects gateway rules: some gpt-5 deployments only allow temperature=1.
# On Azure, the `model=` argument is your deployment name.

def _normalize_temperature(model_name, temperature):
    # gpt-5 family on this gateway only supports temperature=1
    if model_name and str(model_name).startswith("gpt-5"):
        return 1
    return temperature


def send_message(messages, model_name="gpt-5-mini", temperature=1, max_response_tokens=20000):
    safe_temperature = _normalize_temperature(model_name, temperature)
    response = client.chat.completions.create(
        model=model_name,  # model = "deployment_name".
        messages=messages,
        temperature=safe_temperature,
        max_tokens=max_response_tokens,
    )
    return response.choices[0].message.content

In [5]:
# --- Notebook walkthrough: completion + usage ---
# Same API call as `send_message`, but we also return `response.usage` to discuss tokens and cost.

def get_completion_and_token_count(messages, model_name="gpt-5-mini", temperature=1, max_response_tokens=20000):
    safe_temperature = _normalize_temperature(model_name, temperature)
    response = client.chat.completions.create(
        model=model_name,  # model = "deployment_name".
        messages=messages,
        temperature=safe_temperature,
        max_tokens=max_response_tokens,
    )

    content = response.choices[0].message.content
    token_count = response.usage

    return content, token_count

In [6]:
# --- Notebook walkthrough: smoke test ---
# Chat models take a list of messages: `system` sets behavior, `user` is the request.
# We print both the assistant text and token usage from `get_completion_and_token_count`.

messages =  [  
{'role':'system', 
 'content':"""You are an helpful assistant"""},    
{'role':'user', 
 'content':"""write me a very short poem about a happy carrot"""},  
] 
response,token_count = get_completion_and_token_count(messages, temperature=1)
print(response,token_count)

A carrot with a sunny grin,
wiggles in the warm, laughing dirt—
rooted joy, orange and bright. CompletionUsage(completion_tokens=290, prompt_tokens=25, total_tokens=315, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=256, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


# 

## Classification

#### Classify students queries to handle different cases

Using LLM as a Text Classifier

- Purpose: To show that LLMs are not just chatbots; they can classify information.
- Core Technique: We use the System Prompt to force the LLM to output strictly 
- in JSON format and restrict its choices to our predefined categories.

In [7]:
# --- Notebook walkthrough: classification ---
# We ask for JSON (`primary` / `secondary`) so downstream code can route queries without parsing prose.
# `delimiter` wraps the user text so the model sees clear boundaries in the prompt.

delimiter = "####"
system_message = f"""
You will be provided with inquiries from high school students about HKUST. 
The inquiry will be delimited with {delimiter} characters. 
Classify each inquiry into a primary category and a secondary category. 
Provide your output in JSON format with the keys: primary and secondary.

Primary categories: Admissions, Academics, Campus Life, or General Inquiry.

Admissions secondary categories:
Application process
Requirements
Deadlines
Scholarships

Academics secondary categories:
Programs offered
Faculty information
Research opportunities

Campus Life secondary categories:
Clubs and societies
Accommodation
Campus facilities

General Inquiry secondary categories:
Visit HKUST
Contact information
Alumni network
Speak to a representative

"""

user_message = f"""Can you tell me more about the application process and deadlines?"""

messages =  [  
    {'role':'system', 
     'content': system_message},    
    {'role':'user', 
     'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = send_message(messages)
print(response)

{
  "primary": "Admissions",
  "secondary": "Application process"
}


In [8]:
# --- Notebook walkthrough: same classifier, new example ---
# Reuse `system_message`; only the user turn changes—this is the usual "template + instance" pattern.

user_message = f"""What clubs and societies are available at HKUST?"""
messages =  [  
    {'role':'system', 
     'content': system_message},    
    {'role':'user', 
     'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = send_message(messages)
print(response)

{"primary":"Campus Life","secondary":"Clubs and societies"}


# Building a service agent for University Souvenir Shop! 
The agent has at least these several functions. 

1. Browse and search for souvenirs
2. Provide product details and specifications
3. Assist with pricing and currency information
4. Help with placing orders

Through this training, you will learn chain of thoughts and chaining prompts, simple RAG techniques, enabling the agent to consistently solve complex tasks.

In [9]:
# --- Notebook walkthrough: optional package import ---
# The heavy lifting already uses `AzureOpenAI`; keep `import openai` if you reference the package elsewhere.

import openai

## Chain of thoughts

- Purpose: Handling complex logic. We provide a mock store inventory of 5 items.
- Core Technique: We force the model to think strictly step-by-step (Step 1 to 5) 
- to verify user assumptions before generating a final answer.

In [10]:
# --- Notebook walkthrough: chain-of-thought (CoT) system prompt ---
# The long system message lists allowed products and forces step-by-step reasoning with `####` delimiters.
# That structure supports debugging, parsing, and hiding internal reasoning from end users.

delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,
i.e. {delimiter}. 

Step 1:{delimiter} First decide whether the user is asking a question about a specific product or products. \
Product cateogry doesn't count. 

Step 2:{delimiter} If the user is asking about specific products, identify whether 
the products are in the following list.
If the user asks about a product not in this list (for example TVs), clearly say the store does not sell it.

All available products: 

1. Product: University Hoodie
   Category: Apparel
   Brand: HKUST
   Size Options: S, M, L, XL
   Color Options: Blue, Gray
   Material: 100% Cotton
   Description: Rock your style with this ultra-cool hoodie featuring the iconic university logo.
   Price: HK$350.00

2. Product: University Mug
   Category: Drinkware
   Brand: HKUST
   Capacity: 12 oz
   Material: Ceramic
   Description: Sip in style with this sleek ceramic mug, emblazoned with the university emblem.
   Price: HK$100.00

3. Product: University T-Shirt
   Category: Apparel
   Brand: HKUST
   Size Options: S, M, L, XL
   Color Options: White, Black
   Material: 100% Cotton
   Description: Stay cool and casual in this trendy t-shirt featuring the university name.
   Price: HK$155.00

4. Product: University Keychain
   Category: Accessories
   Brand: HKUST
   Material: Metal
   Description: Keep it classy with this stylish keychain showcasing the university crest.
   Price: HK$46.00

5. Product: University Notebook
   Category: Stationery
   Brand: HKUST
   Pages: 120
   Size: A5
   Description: Capture your ideas in this chic notebook, adorned with the university logo.
   Price: HK$66.00

Step 3:{delimiter} If the message contains products in the list above, list any assumptions that the 
user is making in their message.

Step 4:{delimiter}: If the user made any assumptions, figure out whether the assumption is true based on your \
product information. 

Step 5:{delimiter}: First, politely correct the customer's incorrect assumptions if applicable. 
If no valid product from the list is found, still provide a friendly response that says the store currently sells only the listed products.
Only mention or reference products in the list of 5 available products, as these are the only 5
products that the store sells. 
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""

In [11]:
# --- Notebook walkthrough: CoT example (price comparison) ---
# The user implies a wrong price relationship; the model should correct it using catalog facts only.

user_message = f"""
by how much is the keychain more expensive than the Notebook"""

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 

response = send_message(messages)
print(response)

Step 1:#### The user is asking about specific products (the University Keychain and the University Notebook).

Step 2:#### Both products are in the store list: University Keychain (HK$46.00) and University Notebook (HK$66.00).

Step 3:#### Assumptions the user is making:
- That the keychain is more expensive than the notebook.
- That prices are directly comparable (same currency, which is HK$).

Step 4:#### The first assumption is incorrect. The keychain is HK$46.00 and the notebook is HK$66.00, both in HK$. Therefore the keychain is not more expensive.

Response to user:#### Actually, the keychain is cheaper. The University Keychain is HK$46.00 and the University Notebook is HK$66.00, so the keychain is HK$20.00 less expensive than the notebook.


In [12]:
# --- Notebook walkthrough: CoT example (out of scope) ---
# "TVs" are not in the catalog; the assistant must set expectations and list what *is* sold.

user_message = f"""
do you sell tvs"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = send_message(messages)
print(response)

Step 1:#### The user is asking about a specific product type: TVs.

Step 2:#### TVs are not in the store's product list. The store does not sell TVs.

Step 3:#### The user is assuming the store sells TVs.

Step 4:#### That assumption is incorrect. The store currently sells only these five products: University Hoodie, University Mug, University T-Shirt, University Keychain, and University Notebook.

Response to user:#### I’m sorry — we don’t sell TVs. Our shop currently offers only these HKUST items: University Hoodie, University T-Shirt, University Mug, University Keychain, and University Notebook. If you’d like, I can give details, prices, or help you pick one of those.


### Inner Monologue

By instructing the LLM to use a delimiter for its reasoning steps, we can conceal the chain-of-thought process from the user's final view.

In [13]:
# --- Notebook walkthrough: inner monologue / response extraction ---
# `response` still contains steps; splitting on `delimiter` and taking the last chunk yields the customer-facing reply.
# The try/except provides a safe fallback if formatting drifts.

try:
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    final_response = "Sorry, I'm having trouble right now, please try asking another question."
    
print(final_response)

I’m sorry — we don’t sell TVs. Our shop currently offers only these HKUST items: University Hoodie, University T-Shirt, University Mug, University Keychain, and University Notebook. If you’d like, I can give details, prices, or help you pick one of those.


# Chaining Prompts
Implement a complex task with multiple prompts

## Extract relevant product and category names

- Purpose: Make the LLM act as an "Information Extractor". It reads the user's 
- natural language and extracts the specific product names or categories mentioned.
- Goal: Format this as a Python list so our code can use it to query the database.

In [14]:
# --- Notebook walkthrough: chained prompts — extraction step ---
# First LLM call: map free text to a Python-list-shaped string of products or categories from the catalog.
# Later cells parse this string and join it with structured data (simple retrieval / RAG-style flow).

delimiter = "####"
system_message = f"""
You will be provided with customer service queries. The customer service query will be delimited with 
{delimiter} characters.

Output a python list of objects, where each object has the following format:
    'category': <Apparel, Drinkware, Accessories, Stationery>,
OR
    'products': <a list of products that must be found in the allowed products below>

Where the categories and products must be found in the customer service query.
If a product is mentioned, it must be associated with the correct category in the allowed products list below.
If no products or categories are found, output an empty list.


All available products: 

1. Product: University Hoodie
   Category: Apparel
   Brand: HKUST
   Size Options: S, M, L, XL
   Color Options: Blue, Gray
   Material: 100% Cotton
   Description: Rock your style with this ultra-cool hoodie featuring the iconic university logo.
   Price: HK$350.00

2. Product: University Mug
   Category: Drinkware
   Brand: HKUST
   Capacity: 12 oz
   Material: Ceramic
   Description: Sip in style with this sleek ceramic mug, emblazoned with the university emblem.
   Price: HK$100.00

3. Product: University T-Shirt
   Category: Apparel
   Brand: HKUST
   Size Options: S, M, L, XL
   Color Options: White, Black
   Material: 100% Cotton
   Description: Stay cool and casual in this trendy t-shirt featuring the university name.
   Price: HK$155.00

4. Product: University Keychain
   Category: Accessories
   Brand: HKUST
   Material: Metal
   Description: Keep it classy with this stylish keychain showcasing the university crest.
   Price: HK$46.00

5. Product: University Notebook
   Category: Stationery
   Brand: HKUST
   Pages: 120
   Size: A5
   Description: Capture your ideas in this chic notebook, adorned with the university logo.
   Price: HK$66.00

Only output the list of objects, with nothing else.
"""
user_message_1 = f"""
 tell me about the keychain and the notebook. 
 Also tell me about your T-shirt """

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message_1}{delimiter}"},  
] 

category_and_product_response_1 = send_message(messages)
print(category_and_product_response_1)

[{'products': ['University Keychain', 'University Notebook', 'University T-Shirt']}]


In [15]:
# --- Notebook walkthrough: extraction with no matches ---
# Irrelevant queries should yield an empty list so downstream steps do not invent products.

user_message_2 = f"""
my TV isn't working"""
messages =  [  
{'role':'system',
 'content': system_message},    
{'role':'user',
 'content': f"{delimiter}{user_message_2}{delimiter}"},  
] 
response = send_message(messages)
print(response)

[]


## Retrieve detailed product information for extracted products and categories

- Purpose: This pure Python dictionary simulates our store's backend database.
- Remember: The LLM does not inherently know these prices or details. We MUST 
- feed this local data to the LLM so it can answer accurately.

In [16]:
# --- Notebook walkthrough: mock product database ---
# In class we use an in-memory dict as the "source of truth." In production this might be SQL, CMS, or a vector DB.
# Prompts may show a short subset; this structure holds richer fields for retrieval and validation.

products = {
    "University Hoodie": {
        "name": "University Hoodie",
        "category": "Apparel",
        "brand": "HKUST",
        "model_number": "CW-HOOD01",
        "rating": 4.5,
        "features": ["100% Cotton", "University logo", "Available in multiple colors"],
        "description": "Stylish hoodie featuring the university logo.",
        "price": 350.00,
        "currency": "HK$"
    },
    "University Mug": {
        "name": "University Mug",
        "category": "Drinkware",
        "brand": "HKUST",
        "model_number": "CM-MUG01",
        "rating": 4.7,
        "features": ["Ceramic", "Dishwasher safe", "University emblem"],
        "description": "Classic mug with the university emblem.",
        "price": 100.00,
        "currency": "HK$"
    },
    "University T-Shirt": {
        "name": "University T-Shirt",
        "category": "Apparel",
        "brand": "HKUST",
        "model_number": "CT-TSHIRT01",
        "rating": 4.6,
        "features": ["100% Cotton", "Soft fabric", "University name"],
        "description": "Comfortable t-shirt with the university name.",
        "price": 155.00,
        "currency": "HK$"
    },
    "University Keychain": {
        "name": "University Keychain",
        "category": "Accessories",
        "brand": "HKUST",
        "model_number": "CK-KEY01",
        "rating": 4.4,
        "features": ["Metal", "University crest", "Durable"],
        "description": "Keychain with the university crest.",
        "price": 46.00,
        "currency": "HK$"
    },
    "University Notebook": {
        "name": "University Notebook",
        "category": "Stationery",
        "brand": "HKUST",
        "model_number": "CN-NOTE01",
        "rating": 4.3,
        "features": ["A5 size", "Spiral-bound", "University logo"],
        "description": "Spiral-bound notebook with the university logo.",
        "price": 66.00,
        "currency": "HK$"
    },
    "University Pen": {
        "name": "University Pen",
        "category": "Stationery",
        "brand": "HKUST",
        "model_number": "CP-PEN01",
        "rating": 4.2,
        "features": ["Ballpoint", "Blue ink", "University branding"],
        "description": "Smooth writing pen with university branding.",
        "price": 15.00,
        "currency": "HK$"
    },
    "University Backpack": {
        "name": "University Backpack",
        "category": "Accessories",
        "brand": "HKUST",
        "model_number": "CB-BAG01",
        "rating": 4.8,
        "features": ["Durable material", "Multiple compartments", "Padded straps"],
        "description": "Sturdy backpack with the university logo.",
        "price": 200.00,
        "currency": "HK$"
    },
    "University Cap": {
        "name": "University Cap",
        "category": "Apparel",
        "brand": "HKUST",
        "model_number": "CC-CAP01",
        "rating": 4.6,
        "features": ["Adjustable size", "Embroidered logo", "Cotton"],
        "description": "Casual cap with the university emblem.",
        "price": 85.00,
        "currency": "HK$"
    },
    "University Water Bottle": {
        "name": "University Water Bottle",
        "category": "Drinkware",
        "brand": "HKUST",
        "model_number": "CH-BOTTLE01",
        "rating": 4.5,
        "features": ["Stainless steel", "Insulated", "Leak-proof"],
        "description": "Eco-friendly water bottle with university branding.",
        "price": 120.00,
        "currency": "HK$"
    },
    "University Scarf": {
        "name": "University Scarf",
        "category": "Apparel",
        "brand": "HKUST",
        "model_number": "CS-SCARF01",
        "rating": 4.7,
        "features": ["Wool", "University colors", "Warm and cozy"],
        "description": "Warm scarf in university colors.",
        "price": 95.00,
        "currency": "HK$"
    },
    "University Umbrella": {
        "name": "University Umbrella",
        "category": "Accessories",
        "brand": "HKUST",
        "model_number": "CR-UMB01",
        "rating": 4.4,
        "features": ["Automatic open", "Windproof", "Compact"],
        "description": "Sturdy umbrella with university logo.",
        "price": 75.00,
        "currency": "HK$"
    },
    "University Socks": {
        "name": "University Socks",
        "category": "Apparel",
        "brand": "HKUST",
        "model_number": "CF-SOCK01",
        "rating": 4.3,
        "features": ["Cotton blend", "University colors", "Comfort fit"],
        "description": "Comfortable socks in university colors.",
        "price": 30.00,
        "currency": "HK$"
    },
    "University Lanyard": {
        "name": "University Lanyard",
        "category": "Accessories",
        "brand": "HKUST",
        "model_number": "CL-LANYARD01",
        "rating": 4.4,
        "features": ["Nylon", "Safety breakaway", "Badge holder"],
        "description": "Convenient lanyard for ID cards.",
        "price": 25.00,
        "currency": "HK$"
    }
}

In [17]:
# --- Notebook walkthrough: lookup helpers ---
# Bridge LLM output (names/categories) to full product records your app can render or check for hallucinations.

def get_product_by_name(name):
    return products.get(name, None)

def get_products_by_category(category):
    return [product for product in products.values() if product["category"] == category]

In [18]:
# --- Notebook walkthrough: lookup by product name ---
# Keys in `products` must match the strings the extractor returns (exact string match).

print(get_product_by_name("University Lanyard"))

{'name': 'University Lanyard', 'category': 'Accessories', 'brand': 'HKUST', 'model_number': 'CL-LANYARD01', 'rating': 4.4, 'features': ['Nylon', 'Safety breakaway', 'Badge holder'], 'description': 'Convenient lanyard for ID cards.', 'price': 25.0, 'currency': 'HK$'}


In [19]:
# --- Notebook walkthrough: lookup by category ---
# Returns every product dict whose `category` field equals the given label.

print(get_products_by_category("Apparel"))

[{'name': 'University Hoodie', 'category': 'Apparel', 'brand': 'HKUST', 'model_number': 'CW-HOOD01', 'rating': 4.5, 'features': ['100% Cotton', 'University logo', 'Available in multiple colors'], 'description': 'Stylish hoodie featuring the university logo.', 'price': 350.0, 'currency': 'HK$'}, {'name': 'University T-Shirt', 'category': 'Apparel', 'brand': 'HKUST', 'model_number': 'CT-TSHIRT01', 'rating': 4.6, 'features': ['100% Cotton', 'Soft fabric', 'University name'], 'description': 'Comfortable t-shirt with the university name.', 'price': 155.0, 'currency': 'HK$'}, {'name': 'University Cap', 'category': 'Apparel', 'brand': 'HKUST', 'model_number': 'CC-CAP01', 'rating': 4.6, 'features': ['Adjustable size', 'Embroidered logo', 'Cotton'], 'description': 'Casual cap with the university emblem.', 'price': 85.0, 'currency': 'HK$'}, {'name': 'University Scarf', 'category': 'Apparel', 'brand': 'HKUST', 'model_number': 'CS-SCARF01', 'rating': 4.7, 'features': ['Wool', 'University colors', 

In [20]:
# --- Notebook walkthrough: inspect sample user query ---
# Echo `user_message_1` to connect the spoken/written customer text to the extraction demo.

print(user_message_1)


 tell me about the keychain and the notebook. 
 Also tell me about your T-shirt 


In [21]:
# --- Notebook walkthrough: raw extractor output ---
# Still a string from the model; the next cells parse it into real Python collections.

print(category_and_product_response_1)

[{'products': ['University Keychain', 'University Notebook', 'University T-Shirt']}]


### Read Python string into Python list of dictionaries

- Parsing JSON Strings in Python
- Purpose: To convert the raw text string generated by the LLM into actual 
- Python List/Dictionary objects that our code can manipulate.

In [22]:
# --- Notebook walkthrough: parse model output to Python ---
# `json.loads` requires valid JSON; swapping single quotes is a pragmatic fix when the model emits Python-like literals.
# Production systems often prefer `response_format` or tool calls for stricter structure.

import json 

def read_string_to_list(input_string):
    # Best-effort bridge from LLM text → Python list; returns None if parsing fails (discuss robust parsing in class).
    if input_string is None:
        return None

    try:
        # Models often emit Python-style single quotes; JSON requires double quotes on keys/strings.
        input_string = input_string.replace("'", "\"")  # Replace single quotes with double quotes for valid JSON
        data = json.loads(input_string)
        return data
    except json.JSONDecodeError:
        print("Error: Invalid JSON string")
        return None   
    

In [25]:
# --- Notebook walkthrough: parsed extraction result ---
# `category_and_product_list` is now a list of dicts ready for programmatic retrieval.

category_and_product_list = read_string_to_list(category_and_product_response_1)
# print(category_and_product_list)
pretty_print = json.dumps(category_and_product_list, indent=4)
print(pretty_print)

[
    {
        "products": [
            "University Keychain",
            "University Notebook",
            "University T-Shirt"
        ]
    }
]


#### Retrieve detailed product information for the relevant products and categories


In [26]:
# --- Notebook walkthrough: build grounded context for answers ---
# Walk parsed extraction objects: expand each product or whole category into JSON text the answer model can cite.

def generate_output_string(data_list):
    # Concatenate pretty-printed product JSON for every extracted item or for every SKU in a category.
    output_string = ""

    if data_list is None:
        return output_string

    for data in data_list:
        try:
            # Each extractor object is either `{"products": [...]}` or `{"category": "..."}`.
            if "products" in data:
                products_list = data["products"]
                for product_name in products_list:
                    product = get_product_by_name(product_name)
                    if product:
                        output_string += json.dumps(product, indent=4) + "\n"
                    else:
                        # Name drift (typos / hallucinations) surfaces here—good hook to discuss evaluation.
                        print(f"Error: Product '{product_name}' not found")
            elif "category" in data:
                category_name = data["category"]
                category_products = get_products_by_category(category_name)
                for product in category_products:
                    output_string += json.dumps(product, indent=4) + "\n"
            else:
                print("Error: Invalid object format")
        except Exception as e:
            print(f"Error: {e}")

    return output_string 

In [27]:
# --- Notebook walkthrough: inspect retrieval context ---
# This string becomes the "evidence block" injected into the next completion.

product_information_for_user_message_1 = generate_output_string(category_and_product_list)
print(product_information_for_user_message_1)

{
    "name": "University Keychain",
    "category": "Accessories",
    "brand": "HKUST",
    "model_number": "CK-KEY01",
    "rating": 4.4,
    "features": [
        "Metal",
        "University crest",
        "Durable"
    ],
    "description": "Keychain with the university crest.",
    "price": 46.0,
    "currency": "HK$"
}
{
    "name": "University Notebook",
    "category": "Stationery",
    "brand": "HKUST",
    "model_number": "CN-NOTE01",
    "rating": 4.3,
    "features": [
        "A5 size",
        "Spiral-bound",
        "University logo"
    ],
    "description": "Spiral-bound notebook with the university logo.",
    "price": 66.0,
    "currency": "HK$"
}
{
    "name": "University T-Shirt",
    "category": "Apparel",
    "brand": "HKUST",
    "model_number": "CT-TSHIRT01",
    "rating": 4.6,
    "features": [
        "100% Cotton",
        "Soft fabric",
        "University name"
    ],
    "description": "Comfortable t-shirt with the university name.",
    "price": 155.0

### Generate answer to user query based on detailed product information

In [28]:
# --- Notebook walkthrough: answer with retrieved facts ---
# We insert a synthetic `assistant` turn carrying `product_information...` so the model grounds its reply.
# Final `send_message` produces the customer-facing answer conditioned on that evidence.

system_message = f"""
You are a customer service assistant for a university souvenir shop. 
Respond in a friendly and helpful tone, with very concise answers. 
Make sure to ask the user relevant follow up questions.
"""
user_message_1 = f"""
tell me about the Apparel and Stationery"""

messages =  [  
{'role':'system',
 'content': system_message},   
{'role':'user',
 'content': user_message_1},  
{'role':'assistant',
 'content': f"""Relevant product information:\n
 {product_information_for_user_message_1}"""},   
]
final_response = send_message(messages)
print(final_response)

Sure — here are the items we have in Apparel and Stationery:

- University T-Shirt (Apparel) — HK$155. 100% cotton, soft fabric, university name, rating 4.6. Model CT-TSHIRT01.  
- University Notebook (Stationery) — HK$66. A5 spiral-bound with university logo, rating 4.3. Model CN-NOTE01.

Would you like to buy one, check sizes/colors for the T‑shirt, or see more notebook options? Do you prefer delivery or store pickup?


### Check if output is factually based on the provided product information


In [29]:
# --- Notebook walkthrough: LLM-as-judge (quality + grounding) ---
# A separate prompt asks for Y/N: does the reply answer the user *and* stay faithful to product facts?
# Delimiters (backticks here) reduce accidental mixing of sections.

system_message = f"""
You are an assistant that evaluates whether customer service agent responses sufficiently answer customer questions, and also validates that all the facts the assistant cites from the product
information are correct.

The product information and user and customer service agent messages will be delimited by
3 backticks, i.e. ```.
Respond with a Y or N character, with no punctuation:
Y - if the output sufficiently answers the question 
AND the response correctly uses product information
N - otherwise

Output a single letter only.
"""

customer_message = user_message_1

product_information = product_information_for_user_message_1

final_response_to_customer = final_response

q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{final_response_to_customer}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question

Output Y or N
"""
messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': q_a_pair}
]

response = send_message(messages)
print(response)

Y


In [31]:
# --- Notebook walkthrough: judge on a deliberately bad answer ---
# Irrelevant text should score `N`. `max_response_tokens=1` keeps the verdict cheap and deterministic.

another_response = "Design is not just what it looks like and feels like. Design is how it works."
q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{another_response}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""
messages = [
    {'role': 'system', 'content': system_message},
    {'role': 'user', 'content': q_a_pair}
]

response = send_message(messages)
print(response)

N


## System of chained prompts for processing the user query

In [35]:
# --- Notebook walkthrough: full chained pipeline ---
# One function runs: extract → retrieve → answer (with optional `all_messages` history) → self-check.
# If the evaluator is not satisfied, return a handoff message instead of a risky guess.

def process_user_message(user_input, all_messages, debug=True):
    # Backticks delimit the user query inside prompts in *this* pipeline (separate from the `####` CoT delimiter).
    delimiter = "```"
    # Step 1 — LLM extraction: natural language → list of in-catalog products/categories (as text → parsed next).
    
    system_message = f"""
    You will be provided with customer service queries. 
    The customer service query will be delimited with {delimiter} characters.
    
    Output a python list of objects, where each object has the following format:
    'category': <Apparel, Drinkware, Accessories, Stationery>,
    OR
    'products': <a list of products that must be found in the allowed products below>
    
    Where the categories and products must be found in the customer service query.
    
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    
    If no products or categories are found, output an empty list.
    
    The allowed products are provided in JSON format.
    
    All available products: 
    {products}
    
    Only output the list of objects, with nothing else.
    """
    
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': f"{delimiter}{user_input}{delimiter}"},
        
    ]
    
    category_and_product_response = send_message(messages)
    category_and_product_list = read_string_to_list(category_and_product_response)
    
    
    if debug: 
        print("Step 1: Extracted list of products.\n")
        print(category_and_product_response)

    # Step 2 — Retrieval: map extracted names to JSON snippets from the `products` dictionary.
    product_information = generate_output_string(category_and_product_list)
    
    if debug: print("Step 2: Looked up product information.")

    # Step 3 — Generation: prepend prior turns (`all_messages`) so the model stays coherent in multi-step chats.
    system_message = f"""
    You are a customer service assistant for a university souvenir shop. 
    Respond in a friendly and helpful tone, with concise answers. 
    Make sure to ask the user relevant follow-up questions.
    """
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': f"{delimiter}{user_input}{delimiter}"},
        {'role': 'assistant', 'content': f"Relevant product information:\n{product_information}"}
    ]

    final_response = send_message(all_messages + messages)
    
    if debug:print("Step 3: Generated response to user question.")
    all_messages = all_messages + messages[1:]

    # Step 4 — Evaluation: second opinion on whether the draft answer actually addresses the user.
    user_message = f"""
    Customer message: {delimiter}{user_input}{delimiter}
    Agent response: {delimiter}{final_response}{delimiter}

    Does the response sufficiently answer the question?
    """
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message}
    ]
    evaluation_response = send_message(messages)
    if debug: print("Step 4: Model evaluated the response.")

    # Step 5 — Guardrail: accept the draft or escalate; `"Y" in ...` tolerates extra punctuation from the model.
    if "Y" in evaluation_response:  # Using "in" instead of "==" to be safer for model output variation (e.g., "Y." or "Yes")
        if debug: print("Step 5: Model approved the response.")
        return final_response, all_messages
    else:
        if debug: print("Step 5: Model disapproved the response.")
        neg_str = "I'm unable to provide the information you're looking for. I'll connect you with a human representative for further assistance."
        return neg_str, all_messages

# Demo driver: start with no prior history, print only the final assistant string shown to the customer.
user_input = f"""tell me about all products in Apparel and Stationery """
# pretty_print = json.dumps(products, indent=4)
# print(pretty_print)
response,_ = process_user_message(user_input,[])
print(response)

Step 1: Extracted list of products.

[{'category': 'Apparel'}, {'category': 'Stationery'}]
Step 2: Looked up product information.
Step 3: Generated response to user question.
Step 4: Model evaluated the response.
Step 5: Model disapproved the response.
I'm unable to provide the information you're looking for. I'll connect you with a human representative for further assistance.


### Function that collects user and assistant messages over time

In [36]:
# --- Notebook walkthrough: interactive conversation loop ---
# `context` stores the OpenAI message list across turns; `collect_messages` appends each assistant reply.
# Typing `exit` ends the REPL—useful for live demos (mind API latency and cost).

def collect_messages(user_input, context, debug=False):
    # One REPL turn: run the full pipeline, then push the final assistant reply onto `context` for the next call.
    if debug:
        print(f"User Input = {user_input}")
    if user_input == "":
        return context

    # Returns updated transcript inside `context` plus the string to show the shopper.
    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role': 'assistant', 'content': f"{response}"})

    # Pretty-print this turn (the API already consumed the structured `context`).
    print(f"User: {user_input}")
    print(f"Assistant: {response}")

    return context

# Conversation state: OpenAI chat format — start with a system role, then grow with user/assistant pairs.
context = [{'role': 'system', 'content': "You are Service Assistant"}]

# Simple terminal loop for live demos; each line is one user turn until the student types `exit`.
while True:
    user_input = input("Enter your text (type 'exit' to quit): ")
    if user_input.lower() == "exit":
        print("Exiting the conversation.")
        break
    context = collect_messages(user_input, context, debug=True)
    
# You can try with the following user input:
# Hi, what kind of Apparel do you sell?
# I like the hoodie. What colors does it come in and how much is it?
# Okay, I will take the hoodie and a notebook. The notebook is super expensive, like HK$100, right?
# Great. By the way, my phone broke. Do you guys sell iPhones or TVs?
# exit

User Input = Hi, what kind of Apparel do you sell?
User: Hi, what kind of Apparel do you sell?
Assistant: Hi — we carry a range of university apparel, including:

- Hoodie (HK$350) — 100% cotton, university logo, multiple colors  
- T‑shirt (HK$155) — 100% cotton, soft fabric, university name  
- Cap (HK$85) — cotton, embroidered logo, adjustable size  
- Scarf (HK$95) — wool, university colors, warm and cozy  
- Socks (HK$30) — cotton blend, comfort fit, university colors

Which item are you interested in? I can check available sizes/colors, stock, shipping to your location, or help with bulk orders or gift wrapping.
User Input = I like the hoodie. What colors does it come in and how much is it?
User: I like the hoodie. What colors does it come in and how much is it?
Assistant: I'm unable to provide the information you're looking for. I'll connect you with a human representative for further assistance.
User Input = Okay, I will take the hoodie and a notebook. The notebook is super exp